# **Dataset Download & Setup**

In [ ]:
import kagglehub
path = kagglehub.dataset_download("masoudnickparvar/white-blood-cells-dataset")

print("Path to dataset files:", path)

# **Data Loading & Preprocessing**

In [ ]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



base_path = "../kaggle/input/white-blood-cells-dataset"

train_path = base_path + "/Train"
testA_path = base_path + "/Test-A"
testB_path = base_path + "/Test-B"



train_dataset = datasets.ImageFolder(
    root=train_path,
    transform=train_transform
)

testA_dataset = datasets.ImageFolder(
    root=testA_path,
    transform=test_transform
)

testB_dataset = datasets.ImageFolder(
    root=testB_path,
    transform=test_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

testA_loader = DataLoader(
    testA_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

testB_loader = DataLoader(
    testB_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)


print("Train Classes:", train_dataset.classes)
print("Number of Classes:", len(train_dataset.classes))

# **ResNet-50 (Baseline CNN)**

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


model_resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)


num_features = model_resnet.fc.in_features

for param in model_resnet.parameters():
    param.requires_grad = False

model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 5)



model_resnet = model_resnet.to(device)

print(model_resnet)

# **Vision Transformer (ViT)**

In [ ]:
import torch
import torch.nn as nn
from transformers import ViTForImageClassification


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


model_vit = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=5,
    ignore_mismatched_sizes=True
)
for param in model_vit.vit.parameters():
    param.requires_grad = False

model_vit = model_vit.to(device)

print(model_vit)

# **Hybrid CNN + ViT Model (PyTorch Implementation)**

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class HybridCNNViT(nn.Module):
    def __init__(self, num_classes=5, embed_dim=512, num_heads=8, num_layers=2):
        super(HybridCNNViT, self).__init__()

        # CNN backbone (ResNet50)
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-2])

        # Freeze CNN layers
        for param in self.cnn_backbone.parameters():
            param.requires_grad = False

        # Reduce channels
        self.conv_reduce = nn.Conv2d(2048, embed_dim, kernel_size=1)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):


        x = self.cnn_backbone(x)


        x = self.conv_reduce(x)

        B, C, H, W = x.shape


        x = x.view(B, C, H*W)
        x = x.permute(0, 2, 1)


        x = self.transformer(x)

        x = x.mean(dim=1)


        x = self.classifier(x)

        return x

In [ ]:
model_hybrid = HybridCNNViT(num_classes=5)
model_hybrid = model_hybrid.to(device)

print(model_hybrid)

# **Training Loop**

**Training Function (Reused for All Models)**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [ ]:
def train_model(model, train_loader, val_loader, epochs=6, lr=3e-4):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2
    )

    scaler = GradScaler()
    early_stopping = EarlyStopping(patience=3)

    for epoch in range(epochs):


        model.train()
        train_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with autocast():
                outputs = model(images)


                if hasattr(outputs, "logits"):
                    outputs = outputs.logits

                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                with autocast():
                    outputs = model(images)

                    if hasattr(outputs, "logits"):
                        outputs = outputs.logits

                    loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total
        val_loss /= len(val_loader)

        scheduler.step(val_loss)

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"| Train Acc: {train_accuracy:.2f}% "
              f"| Val Acc: {val_accuracy:.2f}%")

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered.")
            break

    return model

In [ ]:
trained_resnet = train_model(
    model_resnet,
    train_loader,
    testA_loader,
    epochs=3
)

In [ ]:
trained_vit = train_model(
    model_vit,
    train_loader,
    testA_loader,
    epochs=3
)

In [ ]:
trained_hybrid = train_model(
    model_hybrid,
    train_loader,
    testA_loader,
    epochs=3
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

In [ ]:
def evaluate_model(model, dataloader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)


            if hasattr(outputs, "logits"):
                outputs = outputs.logits

            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="weighted")
    recall = recall_score(all_labels, all_preds, average="weighted")
    f1 = f1_score(all_labels, all_preds, average="weighted")
    cm = confusion_matrix(all_labels, all_preds)

    print("Accuracy:", acc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("Confusion Matrix:\n", cm)

    return acc, precision, recall, f1, cm

In [ ]:
print("ResNet Results")
evaluate_model(trained_resnet, testA_loader)

In [ ]:
print("ViT Results")
evaluate_model(trained_vit, testA_loader)

In [ ]:
print("Hybrid Model Results")
evaluate_model(trained_hybrid, testA_loader)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_confusion(cm, class_names):

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names,
                yticklabels=class_names,
                cmap="Blues")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()

In [ ]:
acc, precision, recall, f1, cm = evaluate_model(trained_resnet, testA_loader)

plot_confusion(cm, train_dataset.classes)

In [ ]:
acc, precision, recall, f1, cm = evaluate_model(trained_resnet, testA_loader)

plot_confusion(cm, train_dataset.classes)

In [ ]:
acc, precision, recall, f1, cm = evaluate_model(trained_hybrid, testA_loader)

plot_confusion(cm, train_dataset.classes)

In [ ]:
resnet_results = evaluate_model(trained_resnet, testA_loader)
vit_results = evaluate_model(trained_vit, testA_loader)
hybrid_results = evaluate_model(trained_hybrid, testA_loader)

In [ ]:
resnet_acc = resnet_results[0]
vit_acc = vit_results[0]
hybrid_acc = hybrid_results[0]

In [ ]:
import matplotlib.pyplot as plt

models = ["ResNet50", "Vision Transformer", "Hybrid CNN+ViT"]
accuracies = [resnet_acc, vit_acc, hybrid_acc]

plt.figure(figsize=(6,4))
plt.bar(models, accuracies)

plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

models = ["ResNet50", "Vision Transformer", "Hybrid CNN+ViT"]
accuracies = [resnet_acc, vit_acc, hybrid_acc]

plt.figure(figsize=(6,4))
sns.barplot(x=models, y=accuracies)

plt.title("Accuracy Comparison of Models")
plt.ylabel("Accuracy")
plt.xlabel("Model")

plt.show()